In [3]:
import pandas as pd
import os

csv_severity = "CSV/plantvillage_with_severity_todo.csv"
df = pd.read_csv(csv_severity)

print("Severity value counts:")
print(df["severity"].value_counts(dropna=False))

# keep only rows with a severity label (0,1,2,3)
df_labeled = df[df["severity"].notna()].copy()
df_labeled["severity"] = df_labeled["severity"].astype(int)

print("\nLabeled severity counts:")
print(df_labeled["severity"].value_counts())

# we will sample up to N per class
N = 20  # small test

small_dfs = []
for sev in [0, 1, 2, 3]:
    class_df = df_labeled[df_labeled["severity"] == sev]
    sample_n = min(N, len(class_df))
    print(f"Severity {sev}: using {sample_n} samples")
    small_dfs.append(class_df.sample(sample_n, random_state=42))

small_exp_df = pd.concat(small_dfs).reset_index(drop=True)

print("\nTotal samples in small experiment:", len(small_exp_df))
print(small_exp_df["severity"].value_counts())

os.makedirs("CSV", exist_ok=True)
small_csv = "CSV/severity_small_experiment.csv"
small_exp_df.to_csv(small_csv, index=False)
print("\nSaved:", small_csv)


Severity value counts:
severity
0.0    2539
NaN    2117
2.0      65
3.0      44
1.0      34
Name: count, dtype: int64

Labeled severity counts:
severity
0    2539
2      65
3      44
1      34
Name: count, dtype: int64
Severity 0: using 20 samples
Severity 1: using 20 samples
Severity 2: using 20 samples
Severity 3: using 20 samples

Total samples in small experiment: 80
severity
0    20
1    20
2    20
3    20
Name: count, dtype: int64

Saved: CSV/severity_small_experiment.csv


In [4]:
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import pandas as pd

class PlantSeverityDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.df = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]
        label = int(row["severity"])  # 0,1,2,3

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# simple transforms (keep it light for tiny dataset)
transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

dataset = PlantSeverityDataset("CSV/severity_small_experiment.csv", transform=transform)

print("Total samples:", len(dataset))

# tiny train/val split inside the dataset
indices = list(range(len(dataset)))
split = int(0.8 * len(dataset))  # 80 percent train, 20 percent val
train_idx = indices[:split]
val_idx = indices[split:]

train_subset = torch.utils.data.Subset(dataset, train_idx)
val_subset   = torch.utils.data.Subset(dataset, val_idx)

train_loader = DataLoader(train_subset, batch_size=8, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_subset,   batch_size=8, shuffle=False, num_workers=0)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


Total samples: 80
Train batches: 8
Val batches: 2


In [5]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 4 classes: 0,1,2,3
num_classes = 4

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


Using device: cpu


In [6]:
import time
import copy

def train_small(model, criterion, optimizer, train_loader, val_loader, num_epochs=5, device="cpu"):
    best_wts = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
                loader = train_loader
            else:
                model.eval()
                loader = val_loader

            running_loss = 0.0
            running_corrects = 0
            total_samples = 0

            for inputs, labels in loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                batch_size = inputs.size(0)
                running_loss += loss.item() * batch_size
                running_corrects += torch.sum(preds == labels).item()
                total_samples += batch_size

            epoch_loss = running_loss / total_samples
            epoch_acc = running_corrects / total_samples

            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            if phase == "val" and epoch_acc > best_val_acc:
                best_val_acc = epoch_acc
                best_wts = copy.deepcopy(model.state_dict())

    print("\nBest val acc:", best_val_acc)
    model.load_state_dict(best_wts)
    return model

model = train_small(
    model,
    criterion,
    optimizer,
    train_loader,
    val_loader,
    num_epochs=5,
    device=device,
)



Epoch 1/5
------------------------------
train Loss: 1.2811 Acc: 0.3438
val Loss: 1.7383 Acc: 0.0000

Epoch 2/5
------------------------------
train Loss: 0.6130 Acc: 0.7812
val Loss: 1.9424 Acc: 0.0000

Epoch 3/5
------------------------------
train Loss: 0.3016 Acc: 0.9062
val Loss: 1.8185 Acc: 0.0625

Epoch 4/5
------------------------------
train Loss: 0.2699 Acc: 0.8906
val Loss: 1.8975 Acc: 0.0000

Epoch 5/5
------------------------------
train Loss: 0.1932 Acc: 1.0000
val Loss: 1.9127 Acc: 0.0625

Best val acc: 0.0625
